In [71]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os
from torch.utils.data import DataLoader, TensorDataset, random_split

In [ ]:
cwd = os.getcwd()
data_path = os.path.join(cwd, "Data", "ELISa data", "1 lightcurve.csv")
num_csvs = len(os.listdir("C:\\Users\\seanf\\OneDrive\\Documents\\Master's Files\\2nd Year\\Eclipsing Binaries\\Data\\ELISa data"))

# looping through each light curve csv file, adding them all to one 2d array of dimensions (num_csvs x 100)

for i in range(1,3*num_csvs):
    data_path = os.path.join(cwd, "Data", "ELISa data", f"{i} lightcurve.csv")
    if os.path.exists(data_path):
        data = np.loadtxt(data_path, delimiter=',', skiprows=1)
        phases = data[:, 0]
        fluxes = data[:, 1]

        if i == 1:
            fluxes_array = fluxes
        else:
            fluxes_array = np.vstack((fluxes_array, fluxes))

    else:
        continue


In [ ]:
print(fluxes_array.shape) # sanity check


(551, 100)


In [52]:
# Check for available devices and select if available
if torch.cuda.is_available():
    device = torch.device("cuda")       #CUDA GPU
else:
    device = torch.device("cpu")        #if nothing is found use the CPU
print(f"Using device: {device}")

Using device: cpu


In [ ]:
light_curves_tensor = torch.tensor(fluxes_array, dtype=torch.float32)
#light_curve_labels = torch.tensor(labels_array, dtype=torch.long)

In [ ]:
# Splitting synthetic data into training and validation sets
train_size = int(0.8 * fluxes_array.shape[0])
val_size = int(0.2 * fluxes_array.shape[0])

train_data = fluxes_array[:train_size].astype(np.float32)
val_data = fluxes_array[train_size:train_size + val_size].astype(np.float32)

#

# Loading training and validation data into DataLoader for batching
batch_size = 10
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

In [ ]:
# ResNet 50: 
# conv1: 7x7, 64, stride 2. output size 112 x 112
# here the 64 means that the output of this layer will have 64 channels, and the 112 means that the output will have a height and width of 112 pixels.
# conv2_x: 3 x [1x1, 64; 3x3, 64; 1x1, 256], output size 56 x 56
# conv3_x: 4 x [1x1, 128; 3x3, 128; 1x1, 512], output size 28 x 28
# conv4_x: 6 x [1x1, 256; 3x3, 256; 1x1, 1024], output size 14 x 14
# conv5_x: 3 x [1x1, 512; 3x3, 512; 1x1, 2048], output size 7 x 7
# final layer (fully connected): 1x1, 2048, output size 1 x 1

# BUT! our model will be using raw data, not images of the plotted data. Our raw data has length of 100 data points.
# This means we will use 1D convolutional layers instead of 2D convolutional layers.
# So, the architecture will instead be:

# conv1: 7x1, 64, stride 2. output size 50 x 1
# The first 64 here means that the output of this layer will have 64 channels, and the 50 means that the output will have a length of 50 data points.
# 7 is used here as
# conv2_x: 3 x [1x1, 64; 3x1, 64; 1x1, 256], output size 25 x 1
# conv3_x: 4 x [1x1, 128; 3x1, 128; 1x1, 512], output size 13 x 1
# conv4_x: 6 x [1x1, 256; 3x1, 256; 1x1, 1024], output size 7 x 1
# conv5_x: 3 x [1x1, 512; 3x1, 512; 1x1, 2048], output size 4 x 1
# final layer (fully connected): 1x1, 2048, output size 1 x 1

In [ ]:
num_classes = 2 # number of things we are trying to predict.
# in this case, we are trying to classify the eclipsing binary as detached or overcontact, so we have 2 classes.

# simplified cnn, not resnet:

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv1d(in_channels=1, out_channels=64, kernel_size=4, stride=2)
        self.pool = nn.AvgPool1d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=4, stride=2)
        self.fc1 = nn.Linear(128 * 12, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)  # becomes size (batch_size, 1, 100)
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.reshape(x.size(0), -1)  # Flatten the tensor for the fully connected layer
        x = self.fc1(x)
        
        return x

In [77]:
# hyperparameters!

lr = 0.001
epochs = 10
batch_size = 10

In [76]:
model = CNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

In [78]:
train_losses, val_losses = [], []

for epoch in range(epochs):
    model.train()
    train_loss=0
    for inputs in train_loader:
        inputs = inputs.to(device)  # Move inputs to the same device as the model
        optimizer.zero_grad() # set gradients to zero
        predictions = model(inputs) # forward pass
        loss = criterion(predictions, inputs) # calculate loss based on predictions and true labels
        loss.backward() # backpropagation
        optimizer.step() # update weights
        train_loss += loss.item() # accumulate loss for the epoch
    train_losses.append(train_loss / len(train_loader))


RuntimeError: Input type (double) and bias type (float) should be the same